# Bin-picking 6DoF pose estimation: results summary

Side-plate parts in a bin, Isaac Sim data (1,000 frames). Pipeline: **Mask R-CNN detector → GDRN-style RGB-D pose
network (GDRNPP backbone) → ICP on the CAD model → label-free confidence flag**.

All numbers below are recomputed live from `runs/` (training logs and `infer.py` outputs on the held-out scene
`000009`). Re-run top to bottom after retraining or re-running `infer.py`.

**Headline (held-out scene, detector masks):** confident poses ≈ 61 per frame, **99.7 % within 1 mm ADD-S**, centre
error 0.03 mm median, tilt 0.04° median. Confident poses stay available until the bin is empty. *All on
simulated depth; real PhoXi data will be worse (see caveats).*

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":                      # run from the project root
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np, pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Image as IPImage
from PIL import Image

from data_conversion.bin_dataset import BinPoseDataset
from bin6d.config import Cfg, load_config
from bin6d.pose_data import ObjectModel
from bin6d.report import (accuracy_table, detections_table, frame_index, load_frame, load_run, parse_val_log,
                          per_frame_table)
from bin6d.viz import gt_overlay_sets, plotly_scene, render_2d, select

cfg = load_config("configs/default.yaml")
VAL = cfg.data.val_scenes[0]
all_frames = BinPoseDataset(cfg.data.root)
val_frames = BinPoseDataset(cfg.data.root, scenes=[VAL])
model = ObjectModel(val_frames, cfg.data.obj_id, 0, 3000)
pd.set_option("display.width", 200)

# chart styling: reference categorical palette (slots 1-3), recessive grid, one y-axis per plot
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
def style(fig, title, height=360):
    fig.update_layout(template="plotly_white", title=title, height=height, hovermode="x unified",
                      font=dict(color="#52514e"), margin=dict(l=60, r=20, t=60, b=40),
                      legend=dict(orientation="h", y=1.12, x=0))
    fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    return fig
print(f"{len(all_frames)} frames, val scene {VAL}, part diameter {model.diameter:.1f} mm")

## 1. Data

| | |
|---|---|
| Format | BOP (`train_pbr/<scene>/rgb, depth, mask_visib_inst, scene_gt*.json`), 1 shard, 10 scenes × 100 views = **1,000 frames** |
| Sensor | PhoXi-like, 1236×1032, depth in 0.1 mm units; point cloud = back-projected depth (no stored cloud) |
| Labels | 6DoF for **every** part in the bin, including fully occluded ones; instance mask value = GT index + 1 |
| Part | `obj_000001.ply` (mm, recentred), diameter 55.7 mm, 20.9 mm thick, **2-fold symmetric about z** |
| Gotchas | *Picked* parts stay in `scene_gt` parked 1.2–1.8 m outside the bin → the loader drops them. `visib_fract`, `is_topmost`, `n_valid_grasps` are empty. The pile only changes every 10 views → **split by scene**. Metallic parts: ~25 % of visible part pixels have no depth |

Each scene is a picking run: the bin starts full (~280–294 parts) and is emptied in steps.

In [ ]:
idx = frame_index(val_frames)
rows = []
for v in range(0, 100, 10):
    o = val_frames.frame_meta(idx[(VAL, v)])["objects_all"]
    rows.append({"view": v, "parts in bin": len(o["t"]), "visible": int(o["visible"].sum()),
                 "fully hidden": int((~o["visible"]).sum()), ">= 500 visible px": int((o["visib_px"] >= 500).sum())})
display(pd.DataFrame(rows).set_index("view"))

f = load_frame(val_frames, idx[(VAL, 0)])
depth_vis = np.clip((f["depth"] - 600) / 250, 0, 1)
depth_vis = (np.where(f["valid"], 1 - depth_vis, 0) * 255).astype(np.uint8)
rng = np.random.default_rng(0); lut = rng.integers(40, 255, (f["inst"].max() + 1, 3)); lut[0] = 0
tiles = [Image.fromarray(f["rgb"]), Image.fromarray(depth_vis).convert("RGB"), Image.fromarray(lut[f["inst"]].astype(np.uint8))]
strip = Image.new("RGB", (3 * 412, 344))
for i, t in enumerate(tiles):
    strip.paste(t.resize((412, 344)), (412 * i, 0))
display(strip)   # RGB | depth (black = no depth) | visible instance masks

## 2. Pipeline and training setup

| Stage | What | Key settings (`configs/default.yaml`) |
|---|---|---|
| 1. Detector | torchvision Mask R-CNN (ResNet-50 FPN v2, COCO init), 1 class | 640 px native-res crops, **batch 8**, SGD lr 0.02, 40 epochs, up to 400 detections |
| 2. Pose net | GDRN architecture: crop → masks + dense model coordinates + 64 surface regions → Patch-PnP → allocentric 6D rotation + SITE translation | input = RGB + depth (xyz rel. to part surface, valid) + target mask; ConvNeXt-base **initialised from the official GDRNPP ITODD backbone**; **64 crops/batch**, AdamW 2.8e-4, 120 epochs |
| 3. ICP | point-to-plane, observed mask points → camera-facing CAD surface, 8→4→2 mm | rejects corrections > 30° / 15 mm |
| 4. Confidence | label-free, from ICP: accepted **and** fitness ≥ 0.9 **and** fit error ≤ 1.3 × frame median (cap 1.5 mm) | adapts to sensor noise |

**Targets** are ray-cast from the CAD mesh at the GT pose per output pixel (verified vs depth: 0.19 mm median).
**Augmentation (train only, labels stay exact):** virtual camera rotation (roll ±180°, tilt ≤ 8°, exact homography,
0.25 mm label residual), colour/gray/blur/noise, depth noise + random holes, crop jitter, mask dilate/erode;
detector: crops, flips, 90° rotations, colour.
**GPU:** each run capped at 75 % (`gpu_mem_fraction: 0.67`); measured peaks 12.2 GB (detector) and 15.1 GB (pose).

## 3. Training curves (validation scene, evaluated every 2 / 5 epochs)

In [ ]:
det = parse_val_log("runs/detector.log")
fig = go.Figure()
for col, name, c in (("f1", "F1", S1), ("precision", "Precision", S2), ("recall_big", "Recall (≥ 500 px parts)", S3)):
    fig.add_trace(go.Scatter(x=det.index, y=det[col], name=name, mode="lines+markers",
                             line=dict(color=c, width=2), marker=dict(size=8)))
style(fig, "Detector: mask matching at IoU ≥ 0.5").update_yaxes(title="score", range=[0.7, 1.0]).update_xaxes(title="epoch")
fig.show()
print(f"best F1 {det.f1.max():.3f} at epoch {det.f1.iloc[::-1].idxmax()} (saved as runs/detector/best.pt)")

In [ ]:
pose = parse_val_log("runs/pose.log")
fig = make_subplots(rows=1, cols=3, subplot_titles=("ADD-S < 0.1·d (5.6 mm), share", "Rotation error, median (°)",
                                                    "Depth (z) error, median (mm)"))
for i, (col, scale) in enumerate((("add_s_0.1d", 1), ("rot_deg_median", 1), ("z_mm_median", 1)), start=1):
    fig.add_trace(go.Scatter(x=pose.index, y=pose[col] * scale, mode="lines+markers", showlegend=False,
                             line=dict(color=S1, width=2), marker=dict(size=8)), row=1, col=i)
style(fig, "Pose network on GT crops (before ICP)", height=340).update_xaxes(title="epoch")
fig.update_layout(hovermode="closest")
fig.show()
best = pose["add_s_0.1d"].iloc[::-1].idxmax()
print(f"best epoch {best}: " + ", ".join(f"{k}={pose.loc[best, k]:.3f}" for k in ["add_s_0.1d", "rot_deg_median", "z_mm_median"]))

Rotation improves steadily; **depth (z) swings between 4 and 22 mm** until the learning rate decays. GDRN estimates
depth from apparent size (scale-invariant SITE translation), which is built for RGB-only input. This is the
network's weakest component, and the main reason ICP has to reject some poses (see §4).

## 4. End-to-end evaluation on the held-out scene

`infer.py` on every 5th view of scene 000009 (20 frames). **ADD-S** = mean distance between CAD surface points at the
predicted vs GT pose (min over the symmetry); centre error = translation error; tilt = angle between plate normals.
*Ground-truth masks* skips the detector (pose quality alone); *detector masks* is the real pipeline.

In [ ]:
runs = {"detector masks (real pipeline)": "runs/eval_full", "ground-truth masks": "runs/eval_gtmask"}
tables = {name: detections_table(load_run(path, cfg.refine), val_frames, model) for name, path in runs.items()}
for name, df in tables.items():
    print(f"\n{name}: {len(df)} detections, {df.matched.sum()} matched to GT")
    display(accuracy_table(df, diameter=model.diameter))

**Reading it:** the network alone is not picking-accurate (≈ 7 % within 1 mm); its job is to land inside ICP's
capture range. Results are bimodal: where ICP converges, poses are sub-millimetre; where it doesn't, they keep the
network's error. The **confident** subset is what a robot should pick from.

### Confidence flag: fixed vs relative thresholds

In [ ]:
df = tables["detector masks (real pipeline)"]
def variant(min_fit, ratio, cap):
    rc = Cfg(dict(cfg.refine)); rc.confidence_min_fitness, rc.confidence_rmse_ratio, rc.confidence_max_rmse_mm = min_fit, ratio, cap
    d = detections_table(load_run("runs/eval_full", rc), val_frames, model)
    c = d[d.confident & d.matched]
    return {"confident": len(c), "per frame": round(len(c) / d.view.nunique(), 1),
            "ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}", "wrong (>= 1 mm)": int((c.final_add_s >= 1).sum())}
cmp = pd.DataFrame({"fixed 0.8 / 1.0 mm": variant(0.8, 1e9, 1.0), "fixed 0.9 / 0.5 mm": variant(0.9, 1e9, 0.5),
                    "relative 0.9 / 1.3× median (cap 1.5 mm)": variant(0.9, 1.3, 1.5)}).T
display(cmp)

A fixed 0.5 mm fit-error limit sits at the sensor-noise floor: frames with slightly noisier depth lose *all* their
(correct) poses (view 95: 0/13 confident, all correct). The relative limit adapts per frame: same error rate,
more poses.

### Does the robot run out of confident picks as the bin empties?

In [ ]:
pf = per_frame_table(df, val_frames)
display(pf)
fig = go.Figure()
for col, c in (("parts in bin", S1), ("detections", S2), ("confident", S3)):
    fig.add_trace(go.Scatter(x=pf.index, y=pf[col], name=col, mode="lines+markers", line=dict(color=c, width=2),
                             marker=dict(size=8)))
style(fig, "Parts, detections and confident poses per frame (scene 000009)").update_xaxes(title="view (pile changes every 10)").update_yaxes(title="count", rangemode="tozero")
fig.show()

No: 13–88 confident poses per frame all the way down to a nearly empty bin (13 of 13 at the end). A per-pick loop
(capture → detect → pose → ICP → pick the best confident part → re-capture) always has candidates.

### Where do the failures come from?

In [ ]:
m = df[df.matched].copy()
m["depth points"] = pd.cut(m.n_depth_points, [-1, 29, 199, 999, 1e9], labels=["< 30", "30–199", "200–999", "≥ 1000"])
g = m.groupby("depth points", observed=True)
display(pd.DataFrame({"detections": g.size(), "ICP accepted": g.icp_refined.mean().map("{:.0%}".format),
                      "confident": g.confident.mean().map("{:.0%}".format),
                      "final ADD-S < 1 mm": g.final_add_s.apply(lambda s: f"{(s < 1).mean():.0%}")}))
rej = m[~m.icp_refined & (m.n_depth_points >= 30)]
print(f"ICP rejected despite >= 30 depth points: {len(rej)} detections; network error median "
      f"{rej.net_add_s.median():.1f} mm, depth error dominates -> outside ICP's capture range")

## 5. Visual validation

2D overlays: projected 3D boxes of each detection (**green** confident & correct, **red** confident but wrong,
**orange** not confident), xyz axes on confident parts (x red, y green, z blue), **thin white** = GT where the pose
is not ok, **magenta** = visible GT the detector missed. Full bins are dense, so the views are split:

In [ ]:
run = load_run("runs/eval_full", cfg.refine)
def overlay(view, show, width=900):
    f = load_frame(val_frames, idx[(VAL, view)])
    dets = run[(VAL, view)]
    gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
    dets, gt = select(dets, gt, show)
    img = render_2d(f["rgb"], f["K"], dets, model, gt, title=f"view {view}: {show}")
    display(img.resize((width, int(width * img.height / img.width))))
overlay(0, "confident")
overlay(0, "problems")

In [ ]:
overlay(60, "all")
overlay(90, "all")

**Interactive 3D** (drag to rotate, scroll to zoom, hover a box for its fitness / fit error / ADD-S, click legend
items to toggle). For a full-resolution window use `python visualize.py --view 60 --mode 3d`.

In [ ]:
f = load_frame(val_frames, idx[(VAL, 60)])
dets = run[(VAL, 60)]
gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
plotly_scene(f["rgb"], f["xyz"], f["valid"], dets, model, gt, max_points=40000,
             title="view 60: detections (boxes + axes) over the point cloud, camera frame (mm)").show()

## 6. Caveats and next steps

**Caveats**
- **Simulated depth flatters ICP.** Sub-0.1 mm fits come from near-perfect rendered depth. Real PhoXi noise,
  reflections and calibration will lower both precision and the number of confident poses. Re-tune
  `refine.confidence_*` on real captures.
- **Image domain.** The sim renders colour (blue bin); a standard PhoXi gives a grayscale texture. If so, retrain
  with `grayscale_prob: 1.0` in both `aug` sections.
- **Flat-face ambiguity.** A wrong in-plane rotation can still fit the flat face perfectly → high confidence;
  likely source of the remaining wrong confident poses (4 with detector masks, 16 with GT masks, not yet analysed).
- **Pickability is not modelled.** Confident ≠ pickable: suction spot (~6 mm radius), approach clearance and
  pinned parts still need a grasp-selection step.

**Next steps**
1. Run on real PhoXi captures (the biggest unknown).
2. Depth-anchored translation in the pose net (offset from the part's measured surface depth) to bring the
   ~670 "enough depth but ICP rejected" detections into ICP range; ~6 h pose retrain.
3. Ranked pick list: confident parts sorted by fit quality and height, filtered by a suction-feasibility check.